# Day 8 - LLM foundations & API integration practical tasks
***

These tasks cover calling OpenAI API's and prompting.

## Explore how different model parameters and different prompts influence the response.
#### Here are some guidelines and advice you should consider:
* You can add as many cells as you need for each task.
* Use Markdown cells for your conclusions.
* For tasks that do not specify a model, you may choose any of the two available models: `gpt-5.6-luna` / `gpt-5.4-mini`.
* Within each comparison task, change only the model, prompt, or parameter specified in the instructions. Keep everything else - including the model, prompt, and other settings - the same across calls so you can compare the results fairly. For example, when testing different `temperature` values, use the same model, prompt, and output token limit (`max_output_tokens` for the Responses API or `max_completion_tokens` for the Chat Completions API).
* You can use both - [Responses API and Chat Completions API](https://medium.com/@sanathgoutham/openai-responses-api-vs-chat-completions-what-actually-changed-eeee86aff216) - to complete these tasks, but some model parameters are not supported by Responses API.
* LLMs understand [Markdown](https://www.markdownguide.org/cheat-sheet/) formatting in prompts. For example, wrapping text in double asterisks `(**important text**)` marks it as **bold** and can help signal emphasis.
* To copy text from a Markdown cell, open it in edit mode by clicking the pencil icon or double-clicking the cell.
* Optionally, paste API responses into a Markdown cell for better readability.
* Be creative and experiment.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ.pop("OPENAI_API_KEY", None)
load_dotenv()

True

In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in the W2D3 .env file before running this cell.")

client = OpenAI(base_url="https://ds-ai-internship.openai.azure.com/openai/v1/",
                api_key=OPENAI_API_KEY)

#### 1. Write a Python script to call the OpenAI API and generate a response to the prompt below.

Prompt: `"What is artificial intelligence?"`

Use model `gpt-5.6-luna`.

In [3]:
response = client.responses.create(
    model="gpt-5.6-luna",
    input="What is artificial intelligence?",
)

print(response.output_text)

Artificial intelligence (AI) is the field of creating computer systems that can perform tasks commonly associated with human intelligence, such as:

- Understanding language
- Recognizing images or speech
- Learning from data
- Reasoning and solving problems
- Making predictions or decisions
- Generating text, images, music, or code

AI systems can be rule-based, but many modern systems use machine learning: they identify patterns in examples and use those patterns to produce outputs. AI does not necessarily think or understand like a human; its abilities depend on how it was designed, trained, and used.


#### 2. Call the OpenAI API with different models

Prompt `"Explain the concept of machine learning to a 10-year-old."`

`model`: `gpt-5.6-luna` and `gpt-5.4-mini`.

Compare the outputs from each model. 

You can evaluate the differences in:
* Clarity of explanation.
* Simplicity of language (suitable for a 10-year-old).
* Creativity in the response.
* Response time.

In [29]:
import time

MODEL_LUNA = "gpt-5.6-luna"
MODEL_MINI = "gpt-5.4-mini"

prompt_2 = "Explain the concept of machine learning to a 10-year-old."
results_2 = {}

for model in [MODEL_LUNA, MODEL_MINI]:
    start = time.perf_counter()
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt_2}],
    )
    r.elapsed = time.perf_counter() - start
    results_2[model] = r
    print(f"--- {model} ---")
    print(r.choices[0].message.content, "\n")

--- gpt-5.6-luna ---
Machine learning is a way of teaching computers to make decisions by showing them lots of examples instead of giving them every rule.

Imagine you want to teach a computer to recognize cats:

1. You show it many pictures of cats.
2. You also show it pictures of dogs, birds, and other things.
3. The computer looks for patterns—like pointy ears, whiskers, and certain shapes.
4. When it sees a new picture, it makes its best guess: “That looks like a cat!”

The computer may make mistakes at first, but it learns by comparing its guesses with the correct answers and improving over time.

So, **machine learning is like practicing with many examples until you get better at recognizing patterns and making predictions**. 

--- gpt-5.4-mini ---
Machine learning is like teaching a computer to learn from examples, instead of telling it every single rule.

Imagine you want a computer to tell the difference between cats and dogs.  
You could write lots of rules like:

- Cats usua

In [5]:
for model, r in results_2.items():
    text = r.choices[0].message.content
    print(f"{model}: {r.elapsed:.2f}s | {r.usage.completion_tokens} output tokens | {len(text.split())} words")

gpt-5.6-luna: 1.85s | 161 output tokens | 126 words
gpt-5.4-mini: 2.34s | 222 output tokens | 174 words


**My Observation**

**Clarity of explanation:** Both models explained it very clearly. gpt-5.6-luna used one example with 4 simple steps. gpt-5.4-mini also used cats and dogs, but it first showed why writing rules by hand is hard, then gave 4 steps and compared it to a kid learning from flashcards.

**Simplicity of language (suitable for a 10-year-old):** Both are fine for a 10-year-old. gpt-5.4-mini felt a bit friendlier because of the flashcard comparison and the simple "show, find patterns, test, improve" steps. gpt-5.6-luna was also easy and shorter.

**Creativity in the response:** gpt-5.4-mini was more creative. It compared machine learning to flashcards and offered to explain it with a game or sports example. gpt-5.6-luna stayed with one cat example and ended with a practice comparison.

**Response time:** I ran it 3 times and the results were not the same each time: 
- Run 1: gpt-5.6-luna 2.58s, gpt-5.4-mini 2.73s
- Run 2: gpt-5.6-luna 2.64s, gpt-5.4-mini 1.64s
- Run 3: gpt-5.6-luna 1.85s (161 tokens, 126 words), gpt-5.4-mini 2.34s (222 tokens, 174 words)

Each model was faster in a different run, so I can't say that one is always faster. gpt-5.4-mini's answer was longer in two of the runs.

For a 10-year-old I would pick gpt-5.4-mini because of its friendly tone and extra examples. But if I want a shorter and more focused answer, gpt-5.6-luna is better.

#### 3. Call the OpenAI API with different `temperature` values

Prompt: `"Write a two sentence story about a robot learning to cook."`

`temperature` values: `0`, `0.7` and `1.5`. Generate three responses at each temperature.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare the outputs.

Tip: With Chat Completions API you can use parameter `n=3` to generate three different responses to your prompt with one call.

In [6]:
prompt_3 = "Write a two sentence story about a robot learning to cook."
temps = [0, 0.7, 1.5]
results_3 = {}

for t in temps:
    results_3[t] = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": prompt_3}],
        temperature=t,
        n=3,
        reasoning_effort="none",
        max_completion_tokens=100,
    )

In [7]:
for t, r in results_3.items():
    print(f"===== temperature = {t} =====")
    for i, choice in enumerate(r.choices, 1):
        print(f"Answer {i}: {choice.message.content}\n")

===== temperature = 0 =====
Answer 1: The robot carefully studied recipes, its metal fingers clinking as it chopped vegetables and stirred a bubbling pot with precise, mechanical grace. When it finally served dinner, the soup was a little too salty, but the robot glowed with pride because it had made something warm and delicious all by itself.

Answer 2: The robot carefully studied recipes, measuring flour and sugar with precise metal fingers as it tried to bake its first cake. When the cake rose perfectly in the oven, it beeped with joy, discovering that cooking could be both a science and an act of love.

Answer 3: The robot carefully studied recipes, measuring flour and sugar with precise metal fingers as it tried to bake its first cake. When the cake rose perfectly in the oven, it beeped with joy, discovering that cooking could be both a science and an act of love.

===== temperature = 0.7 =====
Answer 1: The robot carefully stirred the soup, consulting its recipe screen and adjust

In [8]:
for t, r in results_3.items():
    texts = [c.message.content for c in r.choices]
    print(f"temperature={t}: {len(set(texts))} unique out of {len(texts)}")

temperature=0: 2 unique out of 3
temperature=0.7: 3 unique out of 3
temperature=1.5: 3 unique out of 3


**My Observation:**

**Temperature 0:** The three stories were almost the same. Answers 2 and 3 were exactly the same (a robot baking its first cake). Answer 1 started with the same words ("The robot carefully studied recipes") but then went to a salty soup. So I got 2 unique answers out of 3, not 1. Even at temperature 0 the model is not 100% fixed.

**Temperature 0.7:** All three stories were different (3 out of 3), but they still felt similar. Each one has the robot studying recipes or stirring soup, with garlic or herbs in the kitchen, and a proud or happy ending. The stories also had different small things, like the chest light glowing, the lesson about patience and curiosity, and the humans smiling. I think this is a good balance between variety and consistency.

**Temperature 1.5:** Again 3 different stories out of 3, but now they started differently too ("The robot whirred into the kitchen...", "The robot clattered into the kitchen with a recipe card...") and had more unexpected details: burned toast and a lopsided soup, a robot programming a note to make it sweeter next time, and a cracked egg with "the tastiest apology". I thought the text would become strange at 1.5, but it was still clear and correct with no random words.

**Conclusion:** A higher temperature gives more variety in details and ideas, and a lower temperature gives similar, repeatable answers. So I would use a low temperature for tasks where the answer should stay the same, and a higher temperature for creative writing.

**Note:** I ran this twice and the stories were different each time, but the pattern was the same: 2, 3 and 3 unique answers at temperatures 0, 0.7 and 1.5.

#### 4. Call the OpenAI API with different `max_output_tokens`/`max_completion_tokens` values.

Prompt: `"Explain the concept of machine learning in simple terms."`

`max_output_tokens` (Response API call)/`max_completion_tokens` (Chat Completions API call) values: `50`, `100` and `200`.

Use `gpt-5.4-mini` for this task.

Compare the length and detail of the responses.

In [9]:
prompt_4 = "Explain the concept of machine learning in simple terms."
token_limits = [50, 100, 200]
results_4 = {}

for limit in token_limits:
    results_4[limit] = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": prompt_4}],
        max_completion_tokens=limit,
        reasoning_effort="none",
    )

In [10]:
for limit, r in results_4.items():
    print(f"===== max_completion_tokens = {limit} =====")
    print(r.choices[0].message.content, "\n")

===== max_completion_tokens = 50 =====
Machine learning is a way for computers to **learn from examples instead of being told every rule explicitly**.

### Simple idea
Normally, you might program a computer with rules like:
- “If the email contains ‘free money,’ mark 

===== max_completion_tokens = 100 =====
Machine learning is a way for computers to **learn from examples** instead of being told every rule step by step.

### Simple idea:
Think of how a child learns to recognize a cat:
- You show many pictures of cats and dogs.
- Over time, the child notices patterns, like ears, whiskers, and shape.
- Then the child can पहचान a new cat they’ve never seen before.

Machine learning works similarly:
- We give a computer lots of data.
- It 

===== max_completion_tokens = 200 =====
Machine learning is a way for computers to **learn from examples** instead of being told every rule by hand.

### Simple idea:
If you show a computer lots of examples, it can start to **notice patterns** and use t

In [11]:
for limit, r in results_4.items():
    c = r.choices[0]
    print(f"limit={limit}: {r.usage.completion_tokens} tokens, finish_reason={c.finish_reason}, {len(c.message.content.split())} words")

limit=50: 50 tokens, finish_reason=length, 38 words
limit=100: 100 tokens, finish_reason=length, 81 words
limit=200: 157 tokens, finish_reason=stop, 123 words


**My Observation:**

**50 tokens:** The answer stopped in the middle of a sentence ("If the email contains 'free money,' mark"). I only got the definition and the start of an example. The `finish_reason` was "length", which means the model reached the limit before it finished.

**100 tokens:** This one had more detail. It gave the definition and the example of a child learning to recognize a cat, and started to explain how machine learning works similarly. But it was cut off again in the middle of a bullet point ("It"). The `finish_reason` was "length" again.

**200 tokens:** This answer was complete. The model stopped on its own after 157 tokens (`finish_reason` = "stop"), so 200 was more than it needed. It had the definition, an example with cat and dog photos, five everyday uses (Netflix or YouTube recommendations, spam filters, voice assistants, face recognition, search results) and a one-line summary.

**Conclusion:** `max_completion_tokens` is only a limit, not the length the model aims for. It doesn't make the answer shorter or better, it just stops the text wherever the limit is, even in the middle of a sentence. To get a short answer that still ends properly, I should ask for it in the prompt (for example "in two sentences") and keep the token limit only as a safety limit.

**Note:** In the 100-token answer the model wrote the Hindi word "पहचान" in the middle of an English sentence ("the child can पहचान a new cat"). It happened in this run and in my first run too, so it's a small mistake the model makes sometimes with this prompt.

#### 5. Call the OpenAI API with different `top_p` values.

Prompt: `"Write a list of 5 completely random and unusual hobbies that no one has ever heard of."`

`top_p` values: `0.3`, `0.7`, and `1.0`.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare response variation as you adjust `top_p`.

In [12]:
prompt_5 = "Write a list of 5 completely random and unusual hobbies that no one has ever heard of."
top_ps = [0.3, 0.7, 1.0]
results_5 = {}

for p in top_ps:
    results_5[p] = []
    for _ in range(3):
        r = client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[{"role": "user", "content": prompt_5}],
            top_p=p,
            reasoning_effort="none",
            max_completion_tokens=250,
        )
        results_5[p].append(r)

In [13]:
for p, runs in results_5.items():
    print(f"===== top_p = {p} =====")
    for i, r in enumerate(runs, 1):
        print(f"--- run {i} ---")
        print(r.choices[0].message.content, "\n")

===== top_p = 0.3 =====
--- run 1 ---
Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow spotting** — tracking and sketching the shapes that clouds’ shadows make on hills, buildings, or water.
2. **Vintage key collecting by sound** — collecting old keys and cataloging them based on the sound they make when tapped.
3. **Moss map making** — creating tiny “maps” or landscapes using different kinds of moss arranged like terrain.
4. **Chair whispering** — sitting in antique chairs and imagining the “stories” they would tell, then writing them as short monologues.
5. **Pebble balancing in impossible places** — balancing stones on unusual surfaces like tree roots, fence posts, or uneven bricks.

If you want, I can make them even weirder or turn them into fake “real” hobby club descriptions. 

--- run 2 ---
Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow spotting** — tracking and sketching the shapes that clouds cast on the ground at different tim

In [14]:
for p, runs in results_5.items():
    line_sets = [
        set(l.strip() for l in r.choices[0].message.content.splitlines() if l.strip())
        for r in runs
    ]
    shared = set.intersection(*line_sets)
    print(f"top_p={p}: lines shared by all 3 runs = {len(shared)}")

top_p=0.3: lines shared by all 3 runs = 2
top_p=0.7: lines shared by all 3 runs = 0
top_p=1.0: lines shared by all 3 runs = 0


**My Observation:**

**Shared lines count:** top_p 0.3 gave 2 shared lines, and 0.7 and 1.0 gave 0. The 2 shared lines at 0.3 are not hobbies. They are the intro sentence and the closing offer ("If you want, I can make them even weirder..."), which were the same in all 3 runs.

**top_p = 0.3:** The 3 runs were very similar. They had the same intro, the same ending and the same layout. "Cloud shadow" was hobby number 1 in all three runs, and "pebble balancing" came up in all three too.

**top_p = 0.7:** There was more variety. The intro changed in two runs and the closing offer was missing in two. But "cloud shadow" came up in all 3 runs again, and "teacup" hobbies came up in all 3 too.

**top_p = 1.0:** This gave the most new ideas, like "Drawer Weather Forecasting", "Muffled opera humming", "Rain puddle genealogy" and "Taxonomy of lost socks". The intros and layout also changed the most. But "cloud shadow" still came up in all 3 runs.

For top_p 1.0, I expected a bigger difference, but the favorite idea ("cloud shadow") still came back in every run.

A higher top_p gave more variety, but the effect was smaller than I expected. Top_p only removes the least likely words, so the model's favorite idea kept coming back even at 1.0. Also, 3 runs for each setting is a small sample. To get really different ideas, I would use a higher temperature together with a higher top_p, or change the prompt.

#### 6. Call the OpenAI API with different `frequency_penalty` values.

Prompt: `"Write a list of funny one-liner jokes about cats."`

`frequency_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the repetition of joke formats changes as you increase the `frequency_penalty`.

In [15]:
prompt_6 = "Write a list of funny one-liner jokes about cats."
fp_values = [0, 0.5, 1.0]
results_6 = {}

for fp in fp_values:
    results_6[fp] = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": prompt_6}],
        frequency_penalty=fp,
        reasoning_effort="none",
        max_completion_tokens=400,
    )

In [16]:
for fp, r in results_6.items():
    print(f"===== frequency_penalty = {fp} =====")
    print(r.choices[0].message.content, "\n")

===== frequency_penalty = 0 =====
Sure — here are some funny cat one-liners:

1. I told my cat to stop acting like a queen, and she replied, “That’s not advice, it’s my title.”
2. My cat doesn’t meow for attention; she files a formal complaint.
3. I work hard so my cat can have a better life.
4. My cat sleeps 18 hours a day, which is more ambition than I have.
5. My cat is on a seafood diet: she sees food, and she eats it.
6. I asked my cat if she wanted to go for a walk, and she laughed in my face.
7. My cat’s favorite exercise is jumping to conclusions.
8. My cat thinks the litter box is a luxury spa.
9. I bought my cat a toy, but she prefers the box like a true minimalist.
10. My cat has two moods: hungry and judging me.
11. My cat isn’t aloof; she’s just selectively friendly.
12. Behind every great cat, there’s a human opening cans.
13. My cat doesn’t need a guard dog — she is the security system.
14. I tried to train my cat, but she said, “You’re lucky I tolerate you.”
15. My cat 

In [17]:
from collections import Counter
import re

skip = {"the", "a", "to", "of", "and", "my", "i", "is", "in", "it", "on", "for", "you", "that", "your", "but", "with"}

for fp, r in results_6.items():
    words = re.findall(r"[a-z']+", r.choices[0].message.content.lower())
    words = [w for w in words if w not in skip]
    print(f"frequency_penalty={fp}:", Counter(words).most_common(5))

frequency_penalty=0: [('cat', 21), ('she', 10), ('s', 6), ('like', 3), ('t', 3)]
frequency_penalty=0.5: [('cat', 13), ('she', 6), ('t', 4), ('s', 4), ('cats', 3)]
frequency_penalty=1.0: [('cat', 10), ('he', 4), ('she', 3), ('ones', 3), ('t', 2)]


**My Observation:**

**Word count:** The word "cat" appeared 21, 13 and 10 times at penalties 0, 0.5 and 1.0. That looks like a big drop, but the lists also got shorter (20, 15 and 10 jokes), so each joke still has about one "cat" in it. So the word count mostly shows that the lists were shorter, not that the repetition was lower.

**Joke format:** This is the better check. I counted how many jokes start with "My cat...":
- Penalty 0: 11 out of 20 (55%)
- Penalty 0.5: 7 out of 15 (47%)
- Penalty 1.0: 4 out of 10 (40%)
So the "My cat..." format got a bit less common as the penalty went up, and other starts like "I asked my cat...", "Cats don't have owners..." and "The cat opened a bakery..." showed up more.

**Joke style:** The style did not follow the penalty in a clear way. At 0 the jokes were mostly about the cat acting like a queen, napping and ignoring me. At 1.0 there were actually more puns ("purr-sonal", "meow-tivational", "purr-fect"), and the cat was sometimes called "he" instead of "she". So I think the changes in style are partly the penalty and partly random.

**Conclusion:** Higher frequency_penalty made the jokes a little less repetitive in how they start, and the lists got shorter in both of my runs. But the effect was small, and the "my cat does X" format stayed the most common one at every setting. The penalty only stops the model from reusing the same words, it doesn't force a different joke structure. To get really different formats, I would ask for them in the prompt (for example "use a different joke style for each one").

#### 7. Call the OpenAI API with different `presence_penalty` values.

Prompt: `"Suggest ten names for a new line of eco-friendly cleaning products."`

`presence_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the diversity of generated names changes as you increase the `presence_penalty`.

In [18]:
prompt_7 = "Suggest ten names for a new line of eco-friendly cleaning products."
pp_values = [0, 0.5, 1.0]
results_7 = {}

for pp in pp_values:
    results_7[pp] = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": prompt_7}],
        presence_penalty=pp,
        reasoning_effort="none",
        max_completion_tokens=300,
    )

In [19]:
for pp, r in results_7.items():
    print(f"===== presence_penalty = {pp} =====")
    print(r.choices[0].message.content, "\n")

===== presence_penalty = 0 =====
Here are ten name ideas for an eco-friendly cleaning product line:

1. **GreenGlint**
2. **EcoSpark**
3. **PureSprout**
4. **NatureNest Clean**
5. **EarthKind**
6. **FreshLeaf Home**
7. **BioBreeze**
8. **ClearCanopy**
9. **Verdant Shine**
10. **CleanCrest Eco**

If you want, I can also suggest names that sound more premium, playful, scientific, or family-friendly. 

===== presence_penalty = 0.5 =====
Here are ten name ideas for an eco-friendly cleaning product line:

1. **GreenGlint**
2. **PureSprout**
3. **EcoShine**
4. **FreshRoot**
5. **CleanLeaf**
6. **BrightBloom**
7. **VerdeVibe**
8. **NatureNook**
9. **SpruceWell**
10. **EarthEcho**

If you want, I can also generate names with a more **luxury**, **playful**, or **scientific/modern** feel. 

===== presence_penalty = 1.0 =====
Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **PureSprout**
3. **EarthLume**
4. **EcoSpark**
5. **CleanKind**
6. **FreshLeaf Cle

In [20]:
import re

for pp, r in results_7.items():
    lines = [
        re.sub(r"^[\d\.\-\*\s]+", "", l).strip()
        for l in r.choices[0].message.content.splitlines()
        if re.match(r"^\s*\d+[\.\)]", l)
    ]
    first_words = [l.split()[0].lower().strip("*:,—-") for l in lines if l.split()]
    print(f"presence_penalty={pp}: {len(lines)} names, {len(set(first_words))} distinct first words")

presence_penalty=0: 10 names, 10 distinct first words
presence_penalty=0.5: 10 names, 10 distinct first words
presence_penalty=1.0: 10 names, 10 distinct first words


**My Observation:**

**First-word check:** All three settings gave 10 names with 10 different first words, so this check doesn't show any difference. The names are mostly one compound word (like "GreenGlint"), so the first word is always different.

**Themes:** The names were very similar at every setting. They are mostly made from the same small group of words: Green, Eco, Pure, Fresh, Leaf, Bloom, Earth, Nature, Verdant and Clean.

**Repeated names:** Many names came back at different settings:
- "PureSprout" appeared at all three (0, 0.5 and 1.0).
- "GreenGlint" appeared at 0 and 0.5.
- "EcoSpark", "Verdant Shine" and "NatureNest Clean" appeared at 0 and 1.0.
- "BrightBloom" appeared at 0.5 and 1.0.
So at 1.0, 5 of the 10 names had already appeared at a lower setting. Only a few new words showed up at 1.0 (Gleam, Lume, Ripple).

**Conclusion:** Raising presence_penalty did not make the names more diverse. The names at 1.0 were not more different than at 0. The penalty only stops the model from reusing a word that already appeared in the same answer, and a list of ten short names is too small for that to matter. I think the effect would be easier to see in a longer text. To get really different name ideas, I would change the prompt (for example, ask for playful, scientific and luxury names) instead of using this parameter.

#### 8. Use the prompt "*Explain the importance of data preprocessing in machine learning.*" with the following system prompts:
1. "You are a university professor teaching a class of beginners."
2. "You are a technical expert explaining to a team of engineers."
3. "You are a comedian explaining this concept in a funny way."

Observe how the tone and style of the response change based on the system prompt.

In [21]:
prompt_8 = "Explain the importance of data preprocessing in machine learning."
system_prompts = {
    "professor": "You are a university professor teaching a class of beginners.",
    "engineer": "You are a technical expert explaining to a team of engineers.",
    "comedian": "You are a comedian explaining this concept in a funny way.",
}
results_8 = {}

for name, sp in system_prompts.items():
    results_8[name] = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": sp},
            {"role": "user", "content": prompt_8},
        ],
        reasoning_effort="none",
        max_completion_tokens=500,
    )

In [22]:
for name, r in results_8.items():
    print(f"===== {name} =====")
    print(r.choices[0].message.content, "\n")

===== professor =====
Data preprocessing is important in machine learning because raw data is usually messy, inconsistent, and not directly usable by algorithms.

Here are the main reasons it matters:

1. **Improves model accuracy**  
   Clean, well-prepared data helps the model learn real patterns instead of noise or errors.

2. **Handles missing or incorrect data**  
   Real-world datasets often have missing values, duplicates, or wrong entries. Preprocessing fixes or removes these problems.

3. **Makes features comparable**  
   Many algorithms work better when numerical values are on a similar scale. For example, scaling helps when one feature ranges from 1 to 10 and another from 1,000 to 1,000,000.

4. **Converts data into usable form**  
   Machine learning models usually require numerical input, so categorical data like “red,” “blue,” or “green” must be encoded.

5. **Reduces noise and redundancy**  
   Preprocessing can remove irrelevant information and improve the signal-to-no

In [23]:
for name, r in results_8.items():
    text = r.choices[0].message.content
    print(f"{name}: {len(text.split())} words, {r.usage.completion_tokens} tokens, finish_reason={r.choices[0].finish_reason}")

professor: 226 words, 325 tokens, finish_reason=stop
engineer: 273 words, 380 tokens, finish_reason=stop
comedian: 301 words, 409 tokens, finish_reason=stop


**My Observation:**

**Professor (beginners):** A numbered list of 7 reasons, each with a short bold title and one or two simple sentences. It uses easy examples, like one feature from 1 to 10 and another from 1,000 to 1,000,000, and turning "red", "blue" and "green" into numbers. It ends with a short summary and an offer to give a simple example. It was the shortest answer: 226 words, 325 tokens.

**Engineer (technical):** Bullet points with sub-points, and more technical words: normalization, standardization, one-hot encoding, label encoding, imputation, outliers, feature selection and convergence for gradient-based methods (like neural networks and logistic regression). It also says bad preprocessing can cause underfitting, overfitting or unstable training, and that in many projects better preprocessing gives bigger gains than changing the model. 273 words, 380 tokens.

**Comedian (funny):** The funniest and the longest one: 301 words, 409 tokens. It starts with "raw data is basically a raccoon with a keyboard", compares bad data to a textbook "in crayon", says a big feature "bullies" a small one, and ends with the model "emotionally spiraling". The useful points (missing data, scaling, encoding) are still there.

**Conclusion:** The system prompt changed the tone, the vocabulary and the examples. The professor used simple examples, the engineer used technical terms, and the comedian used jokes. All three still covered the same main points (cleaning data, missing values, scaling, encoding, faster training). The layout stayed similar too (a "Why it matters" part and a "Common preprocessing steps" list), because my user prompt and the model's usual answer style decided the structure. So a system prompt changes the voice and the audience, but for a very different structure I would need to ask for it directly.

#### 9. Write a Zero-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!".

Use it in your API call.

In [24]:
review = "The product is amazing, and I love using it every day!"

zero_shot_prompt = f"""Classify the sentiment of the following product review as Positive, Negative or Neutral.
Answer with one word only.

Review: "{review}"
Sentiment:"""

r9 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": zero_shot_prompt}],
    temperature=0,
    reasoning_effort="none",
    max_completion_tokens=10,
)
print("Zero-shot:", r9.choices[0].message.content)
print("Input tokens:", r9.usage.prompt_tokens)

Zero-shot: Positive
Input tokens: 47


#### 10. Write a Few-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!"

Use it in your API call.

Observe how the model's performance changes and compare the response to Zero-Shot output.

In [25]:
few_shot_prompt = f"""Classify the sentiment of each product review as Positive, Negative or Neutral.
Answer with one word only.

Review: "Stopped working after two days. Total waste of money."
Sentiment: Negative

Review: "It does what it says. Nothing special, nothing bad."
Sentiment: Neutral

Review: "Great quality and it arrived early, very happy!"
Sentiment: Positive

Review: "{review}"
Sentiment:"""

r10 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": few_shot_prompt}],
    temperature=0,
    reasoning_effort="none",
    max_completion_tokens=10,
)
print("Few-shot:", r10.choices[0].message.content)
print("Input tokens:", r10.usage.prompt_tokens)

Few-shot: Positive
Input tokens: 103


**My Observation:**

**Zero-shot:** The prompt had only the instruction and the review. The model answered "Positive" and used 47 input tokens.

**Few-shot:** I added three examples (Negative, Neutral and Positive). The model gave the same answer, "Positive", but used 103 input tokens, which is 56 more than zero-shot.

**Comparison:** Both gave the correct label, so there is no difference in the result. The review is very clearly positive ("amazing", "love"), so the model doesn't need examples. Few-shot only cost more tokens here.

**Conclusion:** For an easy and obvious review, zero-shot is enough and cheaper. Few-shot is better for harder reviews (sarcasm, mixed feelings, neutral ones) or when the answer must follow a strict format or label set, because the examples show the model exactly what to return.

#### 11. Optimize input tokens.

Prompt: `"As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended."`

Rewrite the prompt to use as few input tokens as possible while preserving its meaning and the requirement to answer in two simple sentences. The response should remain relevant, clear, and accurate.

Run the original and revised prompts using the same model and settings. Compare their input token counts and assess whether shortening the prompt affected the response quality.

You can use the API response’s usage information to check token counts.

In [26]:
original_11 = (
    "As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. "
    "These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. "
    "Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended."
)

short_11 = "In two simple sentences, why would a sci-fi fan suddenly get a romantic comedy recommended?"

res_11 = {}
for name, p in [("original", original_11), ("short", short_11)]:
    res_11[name] = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": p}],
        temperature=0,
        reasoning_effort="none",
        max_completion_tokens=150,
    )
    print(f"===== {name} =====")
    print(res_11[name].choices[0].message.content, "\n")

===== original =====
The recommendation system may have noticed that many people who like sci-fi also enjoy romantic comedies, so it suggests one based on similar users’ tastes. It could also be trying to broaden the user’s choices by recommending something a little different from their usual viewing habits. 

===== short =====
Because recommendation systems often mix in movies with overlapping themes, like witty dialogue, strong chemistry, or a lighthearted tone. They may also be trying to broaden your tastes based on what similar viewers enjoyed. 



In [27]:
for name, r in res_11.items():
    print(f"{name}: {r.usage.prompt_tokens} input tokens, {r.usage.completion_tokens} output tokens")

saved = res_11["original"].usage.prompt_tokens - res_11["short"].usage.prompt_tokens
print("Input tokens saved:", saved)

original: 94 input tokens, 56 output tokens
short: 24 input tokens, 43 output tokens
Input tokens saved: 70


**My Observation:**

**Short prompt:** "In two simple sentences, why would a sci-fi fan suddenly get a romantic comedy recommended?" I removed the background about how recommendation systems work, because the model already knows it, and kept the two important parts: the two-sentence limit and the sci-fi fan getting a romantic comedy.

**Token counts:** The original prompt used 94 input tokens and the short one used 24, so I saved 70 tokens (about 74%). The output was also a bit shorter: 56 tokens for the original and 43 for the short one. The model, temperature and token limit were the same for both.

**Answer quality:** Both answers have two sentences and give a believable reason. The original said that people who like sci-fi often like romantic comedies too, and that the app may be trying to show something different. The short one said that the app mixes in movies with similar themes (witty dialogue, chemistry, a light tone) and may also be broadening my taste based on similar viewers. The short answer gave a bit more detail, but its first sentence starts with "Because", so it reads a little like a fragment.

**Conclusion:** Shortening the prompt cut the input tokens by about three quarters and the answer was still good. The background paragraph had no information the model needed. I removed the filler ("As you know...", "Keeping this background in mind...") and kept the task and the two-sentence rule.

#### 12. Optimize output tokens.

Prompt: `"Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"`

Rewrite the prompt to request a concise, plausible explanation in 10 output tokens or fewer.

Run the original and revised prompts using the same model and settings. Compare their output token counts and check whether the shorter response still provides a meaningful explanation. Refine your prompt if necessary.

You can use the API response’s usage information to check token counts.

In [31]:
prompt_12 = "Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"
final_12 = "Why does a movie app recommend a rom-com to a sci-fi fan? Reply with just the likely cause, max 6 words."

res_12 = {}
for name, p in [("original", prompt_12), ("final", final_12)]:
    res_12[name] = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": p}],
        temperature=0,
        reasoning_effort="none",
        max_completion_tokens=300,
    )
    print(f"===== {name} =====")
    print(res_12[name].choices[0].message.content, "\n")

===== original =====
A movie app can recommend a romantic comedy even if you mostly watch sci-fi for a few common reasons:

- **One-off behavior matters:** If you recently clicked, paused on, searched for, or watched anything adjacent to romance/comedy, the system may have weighted that heavily.
- **Popularity bias:** Apps often surface broadly popular titles, and rom-coms are frequently recommended because they perform well with many users.
- **Collaborative filtering:** The app may be using “people who watched sci-fi also watched this rom-com,” based on overlap with other users rather than your exact taste.
- **Context signals:** Time of day, device, location, or recent activity can influence recommendations.
- **Genre blending:** Some sci-fi titles have romantic or comedic elements, and the system may infer you’re open to mixed-genre content.
- **Cold-start or sparse profile:** If the app doesn’t have enough data about you, it may fall back to general recommendations.
- **Testing/ex

In [32]:
for name, r in res_12.items():
    print(f"{name}: {r.usage.completion_tokens} output tokens, finish_reason={r.choices[0].finish_reason}")

original: 247 output tokens, finish_reason=stop
final: 8 output tokens, finish_reason=stop


**My Observation:**

**Original prompt:** The answer used 247 output tokens. It was a list of 7 possible reasons (one-off behavior, popularity bias, collaborative filtering, context signals, genre blending, cold-start or little data, and A/B testing) and an offer to help more.

**First try:** I asked "Why a rom-com for a sci-fi fan? Answer in 8 words or fewer." The answer was "Because love stories can be delightfully unexpected too." (13 tokens). It is over 10 tokens, and it is not a real explanation, only a nice sentence.

**Improving the prompt:** I asked for the technical cause, told the model to skip the filler, and limited it to 6 words. The results:
- v1: "Because love stories still need a little gravity." - 12 tokens. Too long and only fluff.
- v2: "Collaborative filtering lacked genre-aware features" - 10 tokens. It fits the limit and sounds technical, but it is a guess and doesn't explain this case clearly.
- v3: "Shared audience overlap in recommendations" - 8 tokens. It fits the limit and gives a believable cause: the app recommends what other overlapping audiences also watch.

**Final prompt:** "Why does a movie app recommend a rom-com to a sci-fi fan? Reply with just the likely cause, max 6 words." It used 8 output tokens instead of 247, which is about 97% less, and the answer was the same each time I ran it.

**Trade-off:** The short answer is much cheaper, but I lose the other possible reasons from the long answer (like one-off clicks, popularity or A/B testing), and the wording is a bit technical.

**Conclusion:** Asking for the likely cause with no filler worked better than only asking for a short answer. Just saying "be short" gave a nice sentence without a reason, but asking for the cause gave a useful answer.

#### 13. Expand the given prompt accordingly to achieve the requested output from the list below.

Prompt: `"Explain the importance of exercise." `

Expected output (from three separate prompts):
1. One sentence long
2. Paragraph format
3. Bullet points

In [70]:
base_13 = "Explain the importance of exercise."
prompts_13 = {
    "one sentence": base_13 + " Answer in exactly one sentence.",
    "paragraph": base_13 + " Write a single paragraph of about 5-6 sentences, no lists.",
    "bullet points": base_13 + " Use 5 short bullet points and nothing else.",
}

results_13 = {}
for k, p in prompts_13.items():
    results_13[k] = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": p}],
        temperature=0.5,
        reasoning_effort="none",
        max_completion_tokens=400,
    )

In [71]:
for k, r in results_13.items():
    print(f"===== {k} =====")
    print(r.choices[0].message.content, "\n")

===== one sentence =====
Exercise is important because it strengthens the body, improves heart and mental health, boosts energy and mood, and helps prevent many chronic diseases. 

===== paragraph =====
Exercise is important because it helps keep the body healthy and strong while also supporting mental well-being. Regular physical activity improves heart health, builds muscle and bone strength, and helps maintain a healthy weight. It can also boost energy levels, improve sleep, and reduce the risk of many chronic diseases such as diabetes and high blood pressure. Beyond the physical benefits, exercise can lower stress, improve mood, and increase focus and confidence. Even simple activities like walking, stretching, or cycling can make a meaningful difference when done consistently. Overall, making exercise a regular habit is one of the best ways to support a longer, healthier, and more active life. 

===== bullet points =====
- Improves heart and lung health  
- Builds strength and end

In [72]:
import re

one = results_13["one sentence"].choices[0].message.content
para = results_13["paragraph"].choices[0].message.content
bul = results_13["bullet points"].choices[0].message.content

print("one sentence: sentences =", len(re.findall(r"[.!?](\s|$)", one)))
print("paragraph: sentences =", len(re.findall(r"[.!?](\s|$)", para)), "| has line breaks:", "\n" in para.strip())
print("bullets: count =", len([l for l in bul.splitlines() if l.strip().startswith(("-", "*", "•"))]))

one sentence: sentences = 1
paragraph: sentences = 6 | has line breaks: False
bullets: count = 5


#### 14. Expand this prompt to get more creative responses.

Prompt: `"Explain Artificial Intelligence."`

Expected output (from three separate prompts):
1. Metaphor
2. Story
3. Funny

In [33]:
base_14 = "Explain Artificial Intelligence."
prompts_14 = {
    "metaphor": base_14 + " Do it through one extended metaphor, and keep it under 120 words.",
    "story": base_14 + " Tell it as a short story with a character and a small twist, under 150 words.",
    "funny": base_14 + " Make it genuinely funny, with a joke or two, under 120 words.",
}

results_14 = {}
for k, p in prompts_14.items():
    results_14[k] = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": p}],
        temperature=1,
        reasoning_effort="none",
        max_completion_tokens=400,
    )

In [34]:
for k, r in results_14.items():
    print(f"===== {k} =====")
    print(r.choices[0].message.content, "\n")

===== metaphor =====
Artificial intelligence is like a vast library that doesn’t just store books—it learns how to read them, compare them, and write new pages in the same style. Some shelves are filled with examples: pictures, words, numbers, decisions. The librarian inside the library studies patterns, noticing which pages tend to follow others. Over time, it becomes better at helping visitors: answering questions, spotting mistakes, recommending paths, and even drafting new stories. But it doesn’t “understand” like a person does; it predicts and imitates with remarkable skill. AI is, in essence, a machine that turns experience into useful guesses. 

===== story =====
Mina found a tiny robot in her attic, no bigger than a teacup. “What are you?” she asked.

“I learn from examples,” it said. “Show me enough cats, and I can spot a cat. Show me enough maps, and I can find the fastest path. That is artificial intelligence: machines learning patterns to help make decisions.”

Mina spent t

#### 15. Expand this prompt for different audiences.

Prompt: `"Explain climate change."`

Expected audience (from three separate prompts):
1. Kids
2. General public
3. Politicians

In [35]:
base_15 = "Explain climate change."
prompts_15 = {
    "kids": base_15 + " The audience is children aged 7-9. Use very simple words and a friendly example. Max 100 words.",
    "general public": base_15 + " The audience is the general public with no science background. Be clear and practical. Max 150 words.",
    "politicians": base_15 + " The audience is politicians. Focus on policy options, economic costs and trade-offs, and keep it neutral. Max 150 words.",
}

results_15 = {}
for k, p in prompts_15.items():
    results_15[k] = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": p}],
        temperature=0.5,
        reasoning_effort="none",
        max_completion_tokens=400,
    )

In [36]:
for k, r in results_15.items():
    print(f"===== {k} =====")
    print(r.choices[0].message.content, "\n")

===== kids =====
Climate change means Earth is getting warmer over time. This happens because people burn fuels like coal, oil, and gas, which make a gas called carbon dioxide. This gas traps heat from the sun, like a blanket around Earth.

For example, imagine wearing a thick coat on a hot day. You would feel too warm! Earth can feel that way too.

Climate change can cause hotter days, stronger storms, and less ice at the poles. We can help by saving energy, planting trees, and using less car travel. 

===== general public =====
Climate change means long-term changes in Earth’s weather patterns, especially warming. The main cause today is human activity, mainly burning coal, oil, and gas for electricity, cars, and factories. This releases greenhouse gases like carbon dioxide, which trap heat in the atmosphere, like a blanket around the planet.

As a result, we see hotter days, stronger heatwaves, heavier rain in some places, droughts in others, melting ice, and rising sea levels. Thes

#### 16. Expand this prompt accordingly to instructions below.

Prompt: `"Describe a sunset."`

Expected result (from three separate prompts):

1. Add more details to the prompt, e.g., on what details should it focus on.
2. Constrain the prompt by forbidding it to use words 'sun' and 'light'.
3. Change the output in the style of Shakespeare.

In [37]:
base_16 = "Describe a sunset."
prompts_16 = {
    "detailed": base_16 + " Focus on the colour changes in the sky, the shadows on the ground, the sounds, and the temperature of the air. About 100 words.",
    "no sun/light": base_16 + " You must never use the words 'sun' or 'light' (or any word containing them, like sunset or sunlight). About 80 words.",
    "shakespeare": base_16 + " Write it in the style of Shakespeare, with Early Modern English and a few lines of verse. About 80 words.",
}

results_16 = {}
for k, p in prompts_16.items():
    results_16[k] = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": p}],
        temperature=0.7,
        reasoning_effort="none",
        max_completion_tokens=400,
    )

In [38]:
for k, r in results_16.items():
    print(f"===== {k} =====")
    print(r.choices[0].message.content, "\n")

===== detailed =====
The sunset began as a wash of gold at the horizon, slowly deepening into bands of orange, pink, and soft violet. Higher up, the sky turned a clear, fading blue, then melted into dusky indigo as the light drained away. On the ground, shadows stretched long and thin, turning trees and fences into dark silhouettes. The air cooled gently, losing the warmth of the day and carrying a crisp, still feeling. Birds called more softly now, and the usual daytime noise seemed to settle. A quiet breeze moved through the evening, making the fading world feel calm and peaceful. 

===== no sun/light =====
The sky deepens into layers of amber, rose, and violet, as if a painter has brushed color across the horizon with a slow, careful hand. Clouds gather the day’s warmth and hold it briefly before fading into shadow. Trees stand in dark silhouette, and windows begin to glow from within nearby homes. The air feels softer, quieter, as though evening is drawing a long breath. Everything

In [39]:
text = results_16["no sun/light"].choices[0].message.content.lower()
print("contains 'sun':", "sun" in text, "| contains 'light':", "light" in text)

contains 'sun': False | contains 'light': False


#### 17. Write your observations from tasks 13-16, e.g., how different prompts influence complexity, tone, style and focus of the responses. 

**Observations from tasks 13-16**

**Task 13 (format and length):** The same question gave three different shapes. "One sentence" gave one dense sentence with four benefits. "A paragraph of 5-6 sentences" gave 6 connected sentences with more detail. "5 bullet points" gave 5 short fragments with no explanation. The facts are the same, but the format decides how much detail I get.

**Task 14 (creative styles):** All three kept the main idea (AI learns patterns from data). The metaphor (a library) was easy to picture. The story (Mina and a robot) was the most engaging, and its twist fit the explanation. The funny one was the most direct, with jokes. The style changes the tone, not the facts.

**Task 15 (audience):** For kids it was short and simple with a blanket example. For the general public it was clear and practical, with causes, effects and actions. For politicians it was neutral and focused on policy options and trade-offs. The audience changed the vocabulary, the depth and the examples.

**Task 16 (detail, constraints, style):** The detailed prompt made the model describe exactly the things I listed (colours, shadows, sounds, air). The no "sun" or "light" version worked (my check printed False for both). The Shakespeare version used old words ("doth", "whilst") and rhyming couplets.

**Overall:** The more specific the instruction, the more control I had. Format controls the structure, style controls the tone, the audience controls the vocabulary, and constraints change which words the model picks.

#### 18. Generate a Python code with API call.

Write a prompt that generates a Python function to calculate the factorial of a number. Once you have written the prompt, use it to call the API and generate the code. Test the generated code to ensure it calculates the factorial correctly for various inputs.

In [40]:
import re

prompt_18 = """Write a Python function called factorial(n) that returns the factorial of a non-negative integer n.
Raise a ValueError for negative numbers and a TypeError if n is not an integer.
Return only the code in one python code block, with a short docstring and no extra explanation."""

r18 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": prompt_18}],
    temperature=0,
    reasoning_effort="none",
    max_completion_tokens=400,
)
generated_code = r18.choices[0].message.content
print(generated_code)

def extract_code(text):
    m = re.search(r"```(?:python)?\n(.*?)```", text, re.S)
    return m.group(1) if m else text

```python
def factorial(n):
    """Return the factorial of a non-negative integer."""
    if not isinstance(n, int):
        raise TypeError("n must be an integer")
    if n < 0:
        raise ValueError("n must be non-negative")
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```


In [41]:
import math

namespace = {}
exec(extract_code(generated_code), namespace)
factorial = namespace["factorial"]

for n in [0, 1, 2, 5, 10, 20]:
    assert factorial(n) == math.factorial(n), n
print("Values OK for 0, 1, 2, 5, 10, 20")

for bad, err in [(-1, ValueError), (2.5, TypeError), ("a", TypeError)]:
    try:
        factorial(bad)
        print("No error for", bad)
    except err:
        print(f"{bad!r} correctly raised {err.__name__}")

Values OK for 0, 1, 2, 5, 10, 20
-1 correctly raised ValueError
2.5 correctly raised TypeError
'a' correctly raised TypeError


#### 19. Provide the following buggy Python code to the API and ask it to fix the errors:

```
def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
```

Check if the fixed code is working.

In [43]:
buggy_code = """def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
"""

prompt_19 = f"""Fix the syntax errors in this Python code. Return only the corrected code in one python code block.

{buggy_code}"""

r19 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": prompt_19}],
    temperature=0,
    reasoning_effort="none",
    max_completion_tokens=200,
)
fixed_code = extract_code(r19.choices[0].message.content)
print(fixed_code)

def add_numbers(a, b):
    return a + b

print(add_numbers(5, 10))



In [44]:
exec(fixed_code)

15


#### 20. Summarize this text to 2-3 sentences by using API call.

"Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney."

[<p align="right"> <cite> Source: What is generative AI? </cite></p>](https://www.ibm.com/think/topics/generative-ai)

In [45]:
text_20 = '''Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney.'''

r20 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": "Summarize the following text in 2-3 sentences.\n\n" + text_20}],
    temperature=0.3,
    reasoning_effort="none",
    max_completion_tokens=200,
)
print(r20.choices[0].message.content)

Generative AI uses deep learning models to create new content like text, images, video, and audio based on user prompts, by learning patterns from training data and generating similar but original outputs. Major advances in the field came from VAEs, diffusion models, and especially transformers, which power many popular tools such as ChatGPT, GPT-4, Copilot, BERT, Bard, and Midjourney.


#### 21. Translate the following idioms into a language of your choice, ensuring the meaning is preserved (not just word-for-word translation):

1. "It's raining cats and dogs."
2. "Break a leg."
3. "Kill two birds with one stone."
4. "Spill the beans."
5. "A blessing in disguise."

Try to improve the response as much as possible and evaluate the accuracy of the final output.

In [50]:
idioms = [
    "It's raining cats and dogs.",
    "Break a leg.",
    "Kill two birds with one stone.",
    "Spill the beans.",
    "A blessing in disguise.",
]
LANGUAGE = "Malayalam"
numbered = "\n".join(f"{i}. {s}" for i, s in enumerate(idioms, 1))

v1 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": f"Translate these idioms into {LANGUAGE}:\n{numbered}"}],
    temperature=0,
    reasoning_effort="none",
    max_completion_tokens=400,
)
print("Version 1\n", v1.choices[0].message.content)

Version 1
 Here are natural Malayalam equivalents / translations for those idioms:

1. **It's raining cats and dogs.**  
   **മഴ കനത്തായി പെയ്യുന്നു.**  
   (Literally: It is raining heavily.)

2. **Break a leg.**  
   **നല്ല ഭാഗ്യം നേരുന്നു.**  
   (Used to wish someone good luck, especially before a performance.)

3. **Kill two birds with one stone.**  
   **ഒരു കല്ലുകൊണ്ട് രണ്ട് പക്ഷികളെ വീഴ്‌ത്തുക.**  
   (More natural: **ഒരു ശ്രമത്തിൽ രണ്ട് നേട്ടം നേടുക.**)

4. **Spill the beans.**  
   **രഹസ്യം പുറത്തുപറയുക.**  
   (Literally: reveal the secret.)

5. **A blessing in disguise.**  
   **ആദ്യത്തിൽ ദോഷമായി തോന്നിയെങ്കിലും അവസാനം നന്മയായത്.**  
   (Or: **മറഞ്ഞിരിക്കുന്ന അനുഗ്രഹം.**)

If you want, I can also give you **more idiomatic Malayalam versions** or **simple literal translations** for each.


In [51]:
better_prompt = f"""Translate each English idiom into {LANGUAGE}.
Do NOT translate word for word. Use the natural {LANGUAGE} idiom with the same meaning when one exists.
If there is no equivalent idiom, give a natural phrase with the same meaning.
For each item return: the translation, then a short note (literal meaning in brackets) so I can check it.

{numbered}"""

v2 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": better_prompt}],
    temperature=0,
    reasoning_effort="none",
    max_completion_tokens=600,
)
print("Version 2\n", v2.choices[0].message.content)

Version 2
 1. **കാറ്റും മഴയും ശക്തമായി പെയ്യുന്നു** — [വളരെ ശക്തമായി മഴ പെയ്യുന്നു]  
2. **നല്ല ഭാഗ്യം നേരുന്നു** — [നല്ലത് സംഭവിക്കട്ടെ എന്ന ആശംസ]  
3. **ഒരു കല്ലുകൊണ്ട് രണ്ട് പക്ഷികളെ വീഴ്‌ത്തുക** — [ഒരു പ്രവൃത്തിയിലൂടെ രണ്ട് ലക്ഷ്യങ്ങൾ നേടുക]  
4. **രഹസ്യം പുറത്തുപറയുക** — [ഒളിപ്പിച്ച വിവരം വെളിപ്പെടുത്തുക]  
5. **ദോഷമായി തോന്നിയെങ്കിലും അവസാനം നല്ലതായി മാറിയത്** — [ആദ്യത്തിൽ മോശമായി തോന്നിയെങ്കിലും പിന്നീട് ഗുണകരമായത്]


In [52]:
check_prompt = f"""Review these {LANGUAGE} translations of English idioms. For each one say whether the meaning is preserved and sounds natural to a native speaker, and fix any problem.

Original:
{numbered}

Translations:
{v2.choices[0].message.content}"""

check = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": check_prompt}],
    temperature=0,
    reasoning_effort="none",
    max_completion_tokens=700,
)
print("Review\n", check.choices[0].message.content)

Review
 Here’s a review of each Malayalam translation for **meaning** and **naturalness**, with fixes where needed.

---

### 1. **It's raining cats and dogs.**
**Given:** **കാറ്റും മഴയും ശക്തമായി പെയ്യുന്നു**  
**Meaning preserved?** Yes.  
**Sounds natural?** Mostly yes, but it’s a bit literal and not the most idiomatic Malayalam way to say it.

**Better Malayalam:**
- **കനത്ത മഴ പെയ്യുന്നു**
- **മഴ പെയ്ത് കൊണ്ടിരിക്കുന്നു**
- **വളരെ ശക്തമായി മഴ പെയ്യുന്നു**

**Best choice:** **കനത്ത മഴ പെയ്യുന്നു**  
This is the most natural and concise.

---

### 2. **Break a leg.**
**Given:** **നല്ല ഭാഗ്യം നേരുന്നു**  
**Meaning preserved?** Yes.  
**Sounds natural?** Yes, but it’s a bit formal/neutral. Since the English idiom is used to wish someone luck before a performance, the Malayalam should ideally reflect that context.

**Better Malayalam:**
- **നല്ല ഭാഗ്യം നേരുന്നു**
- **ശുഭാശംസകൾ**
- **നന്നായി ചെയ്യട്ടെ**

**Best choice:** **ശുഭാശംസകൾ**  
If you want the performance context, you could sa

In [53]:
final_prompt = f"""Translate each English idiom into {LANGUAGE}.
For each one, give the most widely used fixed idiom or saying with the same meaning (not a paraphrase, not a literal translation).
If an idiom is vulgar, regional, or has several options, say so in a short note and pick the safest one for general use.
Return a numbered list: the English idiom, the {LANGUAGE} translation, and a short note.

{numbered}"""

final = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": final_prompt}],
    temperature=0,
    reasoning_effort="none",
    max_completion_tokens=700,
)
print("Final version\n", final.choices[0].message.content)

Final version
 1. **It’s raining cats and dogs.** — **കാറ്റും മഴയും ശക്തമായി പെയ്യുന്നു.**  
   **Note:** Malayalam has no very common fixed idiom that exactly matches this English one in everyday general use; this is the safest standard expression.

2. **Break a leg.** — **നല്ല ഭാഗ്യം നേരുന്നു.**  
   **Note:** This is the safest general-use equivalent. In Malayalam, people often simply wish **“all the best”** or **“നല്ല ഭാഗ്യം”** rather than using a fixed idiom.

3. **Kill two birds with one stone.** — **ഒരു കല്ലിൽ രണ്ട് പക്ഷി.**  
   **Note:** This is the widely used Malayalam idiomatic equivalent. It is a direct fixed saying and is commonly understood.

4. **Spill the beans.** — **രഹസ്യം പുറത്തുപറയുക.**  
   **Note:** Malayalam does not have one single universally fixed idiom as common as the English one; this is the safest general-use equivalent.

5. **A blessing in disguise.** — **ദോഷമായി തോന്നിയത് അവസാനം നന്മയായി മാറുക.**  
   **Note:** There is no single very common fixed Malay

**My Observation:**

**Version 1 (plain request):** The answers keep the meaning, but most are plain or word-for-word translations, not real Malayalam idioms (for example, "ഒരു കല്ലുകൊണ്ട് രണ്ട് പക്ഷികളെ വീഴ്ത്തുക" for the two birds one). The grammar in the first answer ("കനത്തായി") also looked a bit odd.

**Version 2 (better prompt + literal note):** The notes were Malayalam explanations, not English word-for-word glosses. Idiom 1 now added the wind ("കാറ്റും മഴയും"), idiom 3 stayed literal, and idiom 5 became an explanation.

**Review pass:** It correctly said idiom 3 is a direct translation and not natural, but its fix was another paraphrase, and it called "മറഞ്ഞ അനുഗ്രഹം" natural although it is a word-for-word translation. The output was also cut off at the token limit, because Malayalam uses many more tokens than English.

**Final version (most widely used fixed idiom + note):** The model said that for idioms 1, 2, 4 and 5 Malayalam has no common fixed idiom, and it gave the plain "safest" translation with a note. For idiom 3 it gave "ഒരു കല്ലിൽ രണ്ട് പക്ഷി" and called it a widely used saying.

**Accuracy:** [Write here what you think as a Malayalam speaker: which translations sound natural, which sound like word-for-word English, and whether "ഒരു കല്ലിൽ രണ്ട് പക്ഷി" is really used.]

**Conclusion:** Malayalam is harder for the model. It mostly gave literal or plain translations, and the improvement steps did not find many real idioms. What helped most was asking for the most widely used fixed idiom and a note, because then the model admitted when there was no equivalent.

#### 22. Generate a paragraph about a cat without using the letter 'e'/'E'.

Write a prompt that forces the AI to produce a three-sentence story about a cat while strictly forbidding it to use letter 'e'.

Use `gpt-5.4-mini` for this task.

Explain why your prompt is effective.

In [60]:
import re

prompt_22 = """Write a three-sentence story about a cat.

ABSOLUTE RULES:
1. The letter "e" (lowercase or uppercase) must NOT appear in any word of your answer. One "e" makes the answer wrong.
2. Every word must be a real English word with correct spelling. Never misspell a word or cut a letter out of it to avoid "e". For example, "gazs" is wrong. If a word needs an "e", pick a different word.

How to do it:
- Write the story in the present tense. Past-tense verbs like "walked", "jumped" or "darted" end in "-ed" and contain "e".
- Never use: the, he, she, her, we, be, was, were, with, then, they, there, here, every, never, even, one, when, because, each, very.
- Prefer words like: a, this, that, its, his, my, and, but, on, in, at, to, from, by, as, if, so, up, out, all, now, soon, always.
- Check every single word twice: once for the letter "e", and once to make sure it is spelled correctly.

Output only the three sentences. No title, no extra text."""

def words_with_e(text):
    return [w for w in re.findall(r"[A-Za-z']+", text) if "e" in w.lower()]

messages = [{"role": "user", "content": prompt_22}]
for attempt in range(1, 6):
    r22 = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=messages,
        max_completion_tokens=800,
    )
    story = r22.choices[0].message.content.strip()
    bad = words_with_e(story)
    print(f"Attempt {attempt}:\n{story}\nWords with 'e': {bad}\n")
    if not bad:
        break
    messages.append({"role": "assistant", "content": story})
    messages.append({"role": "user", "content": f"These words contain the letter 'e': {', '.join(bad)}. Rewrite the whole story, still three sentences, with no 'e' anywhere and every word spelled correctly. Output only the story."})

Attempt 1:
A cat sits on a warm mat, and its tail taps soft, slow drums.  
A small moth swoops by, and a cat paws at it with swift joy.  
Soon, a cat curls up in a sun patch and hums in a soft nap.
Words with 'e': []



**Observation**
- The strict prompt worked on the first attempt: the code found no "e" in any word.
- The story has 3 sentences, all in the present tense.
- This time every word is spelled correctly. The earlier run had "gazs", a broken spelling.

**Why the prompt works**
- It says one "e" makes the answer wrong, so the rule is clear and strict.
- It has a separate spelling rule, with "gazs" as an example of what not to do. This stopped the model from cutting letters out of words.
- It asks for present tense, because past-tense words like "walked" and "darted" end in "-ed" and have an "e".
- It bans common words with "e" (the, she, with, they...) and gives safe words to use instead.
- It asks the model to check each word twice: once for "e" and once for spelling.

**Conclusion**
- A strict prompt with clear rules, an example of a wrong answer, and safe words to use gives much better results than a simple prompt.
- The model sees tokens, not letters, so it can still fail. I check the answer in code and retry with the exact bad words to be safe.
- The code only checks for the letter "e", so I also read the story to check the spelling.

#### 23. Extract JSON Data from the input text in two different ways.

Input Text: "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

Extract the information from the text above using two different methods:

1. Write a prompt asking the AI to return a JSON object with the fields `destination`, `start_date`, and `stay_duration`.

2. Instead of putting rules in the prompt, provide a specific JSON schema that defines the field names and their required data types `destination` (the first city), `start_date` (formatted as YYYY-MM-DD), and `stay_duration` (the number of nights as an integer).

Which method yields a more reliable result for a computer program to process, and why?

In [61]:
import json

text_23 = "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

prompt_23a = f"""Extract the trip details from the text and return ONLY a JSON object with the fields
destination, start_date and stay_duration. Use the first city as destination, YYYY-MM-DD for start_date (assume the year 2026),
and the number of nights as an integer for stay_duration.

Text: "{text_23}" """

r23a = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": prompt_23a}],
    temperature=0,
    reasoning_effort="none",
    max_completion_tokens=150,
)
raw_a = r23a.choices[0].message.content
print("Raw output (method 1):")
print(raw_a)

try:
    print("Parsed:", json.loads(raw_a))
except json.JSONDecodeError as e:
    print("Not valid JSON as returned:", e)

Raw output (method 1):
{"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5}
Parsed: {'destination': 'Tokyo', 'start_date': '2026-06-12', 'stay_duration': 5}


In [62]:
schema = {
    "name": "trip_details",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "destination": {"type": "string"},
            "start_date": {"type": "string", "description": "YYYY-MM-DD, assume the year 2026"},
            "stay_duration": {"type": "integer", "description": "number of nights"},
        },
        "required": ["destination", "start_date", "stay_duration"],
        "additionalProperties": False,
    },
}

r23b = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": f"Extract the trip details from this text (assume the year 2026): {text_23}"}],
    temperature=0,
    reasoning_effort="none",
    max_completion_tokens=150,
    response_format={"type": "json_schema", "json_schema": schema},
)
data = json.loads(r23b.choices[0].message.content)
print(data)
print({k: type(v).__name__ for k, v in data.items()})

{'destination': 'Tokyo', 'start_date': '2026-06-12', 'stay_duration': 5}
{'destination': 'str', 'start_date': 'str', 'stay_duration': 'int'}


**Which method is more reliable**

**Results**
- Both methods gave the same output: `{'destination': 'Tokyo', 'start_date': '2026-06-12', 'stay_duration': 5}`.
- The types are correct: two strings and `stay_duration` as an integer.
- The date is in YYYY-MM-DD format.

**Method 1 (rules in the prompt)**
- It worked this time and the JSON could be parsed.
- Nothing forces it. The model could add a sentence before the JSON, use a code block, write "5 nights" as a string, or change a field name. Any of these breaks a program.

**Method 2 (JSON schema, strict mode)**
- The API guarantees valid JSON with exactly these 3 fields and these types, and no extra fields.
- The program can parse it without any cleaning.

**Conclusion**
- The schema method is more reliable. Both matched here, but the prompt only works because the model follows instructions, which is usually fine but not guaranteed. The schema is enforced by the API.
- The schema only fixes the format, not the facts. The text had no year, so I had to tell the model to assume 2026 in both methods.

#### 24. Create a structured table comparing different programming languages.

Write a prompt that generates a table in this format (example below) with accurate information.

| Language    | Use Case               | Advantages                     | Disadvantages                |
|-------------|------------------------|---------------------------------|------------------------------|
| Python      | Data Science, AI       | Easy to learn, versatile       | Slower execution speed       |
| Java        | Enterprise Applications| Platform-independent, robust   | Verbose syntax               |
| JavaScript  | Web Development        | Fast, widely supported         | Limited for backend tasks    |

**Tip**: To enhance readability, you can copy and paste the generated table into a Markdown cell.

In [64]:
from IPython.display import Markdown, display

prompt_24 = """Create a Markdown table comparing 8 popular programming languages with exactly these four columns:
| Language | Use Case | Advantages | Disadvantages |

Include Python, Java, JavaScript, C++, C#, Go, Rust and PHP.
Keep every cell short (max 8 words) and make sure the information is accurate and not generic.
Return only the table."""

r24 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": prompt_24}],
    temperature=0,
    reasoning_effort="none",
    max_completion_tokens=700,
)
table = r24.choices[0].message.content
#print(table)
display(Markdown(table))

| Language | Use Case | Advantages | Disadvantages |
|---|---|---|---|
| Python | Data science, automation, web APIs | Huge ecosystem, readable syntax | Slower runtime, GIL limits threads |
| Java | Enterprise backends, Android apps | Strong tooling, portable JVM bytecode | Verbose syntax, higher memory use |
| JavaScript | Web frontends, Node.js servers | Runs everywhere, async-friendly | Dynamic quirks, callback complexity |
| C++ | Game engines, systems software | Fast, fine-grained hardware control | Complex memory bugs, steep learning curve |
| C# | Windows apps, Unity games | Excellent tooling, modern language features | Best on Microsoft stack |
| Go | Cloud services, DevOps tools | Simple concurrency, fast compilation | Limited generics history, fewer abstractions |
| Rust | Systems programming, safe tooling | Memory safety without GC | Steep borrow checker learning curve |
| PHP | Server-side websites, CMS platforms | Easy deployment, huge hosting support | Inconsistent standard library, legacy baggage |

**My Observation**

The model gave a clean table with 4 columns and 8 languages. The cells are short and easy to scan, and each language has its own use case, advantages and disadvantages. Naming the columns in the prompt gives a clear, consistent table.

#### 25. Generate a quiz about machine learning basics.

Look at the example below and write a prompt that generates a quiz with identical formatting. Ensure the title formatting, list markers (a, b, c, d), and the correct answer indicator (*) are accurately replicated.

**Machine learning basics quiz**

1. Which of the following best describes supervised learning?  
a) The model learns from unlabeled data to find hidden patterns  
b) The model learns from labeled data to make predictions *  
c) The model is programmed with explicit rules  
d) The model randomly guesses outputs without training

2. Which of these is a common application of machine learning in everyday life?  
a) Sorting files manually on a computer  
b) Email spam filtering *  
c) Writing code by hand  
d) Using a calculator for arithmetic

3. What is overfitting in machine learning?  
a) When the model performs well on new, unseen data  
b) When the model is too simple and underperforms  
c) When the model learns the training data too well, including noise, reducing generalization *  
d) When the model runs faster than expected

4. Which ethical concern is commonly associated with machine learning algorithms?  
a) Increased hardware costs  
b) Bias in training data leading to unfair predictions *  
c) The inability to process large datasets  
d) Lack of available programming languages

5. Which of the following is NOT a type of machine learning?  
a) Supervised learning  
b) Reinforcement learning  
c) Unsupervised learning  
d) Manual learning *

In [65]:
from IPython.display import Markdown, display

prompt_25 = """Generate a multiple-choice quiz with 5 questions about machine learning basics.
Follow this format EXACTLY:

**Machine learning basics quiz**

1. Question text?  
a) option  
b) option  
c) option  
d) option

Rules:
- The title is bold on the first line, exactly "Machine learning basics quiz".
- Questions are numbered 1. to 5.
- Options are lettered a), b), c), d), each on its own line, with two spaces at the end of the question line and of each option line so Markdown keeps the line breaks.
- Put a single asterisk (*) after the correct option, with a space before it. Mark exactly one option per question.
- Vary the position of the correct answer.
- Cover: supervised learning, an everyday application, overfitting, an ethical concern, and one 'which is NOT a type of machine learning' question.
- Do not add anything before or after the quiz."""

r25 = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": prompt_25}],
    temperature=0.4,
    reasoning_effort="none",
    max_completion_tokens=900,
)
quiz = r25.choices[0].message.content
#print(quiz)
print("\nQuestions with a marked answer:", sum(1 for l in quiz.splitlines() if l.rstrip().endswith(" *")))
display(Markdown(quiz))


Questions with a marked answer: 5


**Machine learning basics quiz**

1. Which type of machine learning uses labeled examples to learn a mapping from inputs to outputs?  
a) Unsupervised learning  
b) Supervised learning *  
c) Reinforcement learning  
d) Self-training

2. Which is an everyday application of machine learning?  
a) A calculator adding two numbers  
b) A thermostat set to a fixed temperature  
c) A spam email filter *  
d) A light switch turning on a bulb

3. What does overfitting mean?  
a) The model performs well on training data but poorly on new data *  
b) The model ignores all patterns in the data  
c) The model uses too little data to train  
d) The model always improves with more features

4. Which is an ethical concern in machine learning?  
a) Bias in training data *  
b) Using a keyboard shortcut  
c) Faster internet speed  
d) Larger file storage

5. Which is NOT a type of machine learning?  
a) Supervised learning  
b) Reinforcement learning  
c) Unsupervised learning  
d) Spreadsheet learning *

**My Observation**

The model gave a quiz with 5 questions and 4 options (a to d) for each. The correct answer is marked with *. The questions cover different basic questions.All the marked answers are correct, and the wrong options are clearly wrong but still make sense as choices. The correct answer is not always in the same place . Giving the topics and the exact format in the prompt gives a clean quiz that is easy to read.

#### 26. Create a comprehensive online course outline for beginners.

Write a prompt that generates entire course outline, following the example provided below. Ensure the prompt is detailed enough to cover all sections, with clear formatting and practical guidance. 

Sections to include in course outline:
- Course overview - description, objectives, target audience.
- Module breakdown - 5 modules with title, short description, three key learning outcomes and an interactive assignment in the end of each module.
- Assessment plan - a table outlining the assessment methods with description and weighting.
- Certification criteria - requirements to earn a certificate.
- Marketing strategy - step-by-step guide for promoting the course.

Include a detailed description of the type of information each section should contain. Maintain the **same formatting**, **section titles**, and **course name**. While the generated descriptions, modules and marketing strategy steps may differ and incorporate new ideas and concepts, it must remain aligned with the theme of the task.

Put the final output of your API call in a Markdown cell. 

#### Example:
---
---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course is designed to equip learners with the foundational skills and knowledge required to craft effective prompts for AI applications. By mastering prompt engineering, students will learn how to optimize AI models for various tasks, including text generation, data analysis, and creative problem-solving. The course is ideal for individuals interested in AI, developers, data scientists, and anyone looking to leverage AI tools in their personal or professional projects.

#### Objectives
- Understand the principles of prompt engineering and its role in AI workflows.
- Learn techniques to design, test, and refine prompts for optimal AI performance.
- Apply prompt engineering skills to real-world scenarios and projects.

#### Target Audience
- Beginners with no prior experience in AI or programming.
- Professionals seeking to integrate AI into their workflows.
- Students and enthusiasts curious about AI capabilities.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module introduces the concept of prompt engineering, its importance in AI applications, and the basics of interacting with AI models.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its significance in AI workflows.
  2. Understand the structure and components of effective prompts.
  3. Explore real-world use cases of prompt engineering.
- **Interactive Assignment:** Write a simple prompt for a text-generation AI model to generate a short story.

---

#### Module 2: Principles of Effective Prompt Design
- **Description:** Learn the core principles and techniques for designing prompts that yield accurate and relevant AI outputs.
- **Key Learning Outcomes:**
  1. Identify factors that influence AI responses (e.g., context, specificity, tone).
  2. Explore strategies for iterative prompt refinement.
  3. Understand common pitfalls and how to avoid them.
- **Interactive Assignment:** Design and test three variations of a prompt for a chatbot to answer customer queries effectively.

---

#### Module 3: Advanced Techniques in Prompt Engineering
- **Description:** Dive deeper into advanced methods, including chaining prompts, dynamic prompts, and leveraging AI tools for complex tasks.
- **Key Learning Outcomes:**
  1. Learn how to chain prompts for multi-step tasks.
  2. Explore dynamic prompt techniques for adaptive AI responses.
  3. Understand how to integrate external data sources into prompts.
- **Interactive Assignment:** Create a multi-step prompt to generate a detailed report based on user-provided data.

---

#### Module 4: Applications of Prompt Engineering Across Industries
- **Description:** Discover how prompt engineering is applied in various industries, such as healthcare, education, marketing, and software development.
- **Key Learning Outcomes:**
  1. Analyze industry-specific use cases of prompt engineering.
  2. Learn how to tailor prompts for different domains and objectives.
  3. Explore ethical considerations in prompt engineering.
- **Interactive Assignment:** Develop a prompt for a marketing campaign, including product descriptions and social media posts.

---

#### Module 5: Building and Testing AI Projects with Prompt Engineering
- **Description:** Apply your skills to design, test, and deploy AI projects using prompt engineering techniques.
- **Key Learning Outcomes:**
  1. Plan and execute an AI project using prompt engineering.
  2. Test and refine prompts for optimal project outcomes.
  3. Document and present your project effectively.
- **Interactive Assignment:** Create a mini-project using prompt engineering to solve a real-world problem (e.g., automating customer support).

---

### Assessment Plan

| Assessment Method      | Description                          | Weightage (%) |
|------------------------|--------------------------------------|---------------|
| Quizzes               | Multiple-choice quizzes after each module | 30%          |
| Assignments           | Practical exercises for each module  | 40%           |
| Final Project         | A comprehensive project combining all skills | 20%          |
| Participation         | Active participation in discussions  | 10%           |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum score of 70% across all assessments.
2. Submit all interactive assignments and the final project.
3. Actively participate in at least 80% of discussion forums and live sessions.

---

### Marketing Strategy

#### Step 1: Social Media Campaigns
- Create engaging posts highlighting the course benefits, modules, and outcomes.
- Share testimonials and success stories from beta testers or early participants.
- Use targeted ads on platforms like LinkedIn, Facebook, and Instagram.

#### Step 2: Email Marketing
- Build an email list of potential learners (e.g., professionals, students).
- Send a series of emails introducing the course, its benefits, and enrollment deadlines.
- Include exclusive early-bird discounts and referral incentives.

#### Step 3: Partnerships
- Collaborate with universities, coding bootcamps, and AI communities to promote the course.
- Partner with influencers and thought leaders in the AI space for endorsements.

#### Step 4: Content Creation
- Publish blog posts, videos, and webinars on topics related to prompt engineering.
- Share free resources (e.g., mini-guides, templates) to attract interest.
- Host live Q&A sessions to engage with the audience and answer questions about the course.

#### Step 5: Testimonials and Reviews
- Collect feedback from early participants and showcase their success stories.
- Encourage learners to share their experiences on social media and review platforms.

---

In [69]:
from IPython.display import Markdown, display

COURSE = "AI Agents and Tool Calling in Practice"

prompt_26 = f"""Create a complete course outline for a beginner-friendly course titled "{COURSE}".

Use exactly this structure and Markdown formatting:

---

## Course Outline: {COURSE}

---

### Course Overview
A short paragraph about the course.

#### Objectives
3 bullet points.

#### Target Audience
3 bullet points.

---

### Module Breakdown
Exactly 5 modules, each written as "#### Module N: Title" with:
- **Description:** 2 sentences.
- **Key Learning Outcomes:** a numbered list of exactly 3 outcomes.
- **Interactive Assignment:** one practical assignment.
The last module must be a final project.

---

### Assessment Plan
A Markdown table with the columns: Assessment Method | Description | Weightage (%).
Use exactly 4 rows. The weightages must add up to exactly 100.

---

### Certification Criteria
A numbered list of exactly 3 requirements to earn a certificate.

---

### Marketing Strategy
Exactly 5 steps, each written as "#### Step N: Title" with 2 to 3 bullet points.

Formatting rules:
- Put a horizontal rule (---) before the title, after the title, and between all major sections as shown above.
- Output only the outline in Markdown. No text before or after it."""

r26 = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{"role": "user", "content": prompt_26}],
    max_completion_tokens=4000,
)

outline = r26.choices[0].message.content.strip()

# make sure there is always a line above the title, even if the model skips it
if not outline.startswith("---"):
    outline = "---\n\n" + outline

print("finish_reason:", r26.choices[0].finish_reason)
display(Markdown(outline))

finish_reason: stop


---

## Course Outline: AI Agents and Tool Calling in Practice

---

### Course Overview
This beginner-friendly course introduces the core concepts behind AI agents, tool calling, and workflow automation. Learners will build practical agent-based applications that use language models, external tools, APIs, memory, and structured decision-making.

#### Objectives
- Understand how AI agents reason, plan, and interact with tools.
- Build applications that connect language models to APIs and external services.
- Design, test, and evaluate reliable agent workflows.

#### Target Audience
- Beginners interested in AI applications and automation.
- Developers with basic programming experience who want to work with AI agents.
- Product managers, analysts, and technical professionals exploring tool-enabled AI.

---

### Module Breakdown

#### Module 1: Foundations of AI Agents
- **Description:** This module introduces AI agents and explains how they differ from traditional chatbots and scripted programs. Learners will explore agent components such as models, prompts, tools, memory, and decision loops.
- **Key Learning Outcomes:**
  1. Define AI agents and identify their core components.
  2. Explain how agents use reasoning and decision-making loops.
  3. Distinguish between chat-based, workflow-based, and autonomous agent systems.
- **Interactive Assignment:** Map the components and execution flow of a real-world AI agent, such as a customer support assistant.

#### Module 2: Tool Calling Fundamentals
- **Description:** Learners will discover how language models select and call external tools using structured inputs and outputs. The module covers tool schemas, function definitions, validation, and handling tool responses.
- **Key Learning Outcomes:**
  1. Create tool definitions with clear names, descriptions, and parameters.
  2. Implement a basic tool-calling workflow.
  3. Validate tool inputs and format tool outputs for model use.
- **Interactive Assignment:** Build a tool-enabled assistant that can perform calculations, retrieve information, and return structured responses.

#### Module 3: Connecting Agents to APIs and Data
- **Description:** This module focuses on integrating agents with external APIs, databases, and application services. Learners will practice authentication, error handling, data transformation, and safe access to external resources.
- **Key Learning Outcomes:**
  1. Connect an agent to a REST API or database.
  2. Handle authentication, failures, timeouts, and unexpected responses.
  3. Transform external data into useful context for an AI agent.
- **Interactive Assignment:** Create an agent that uses a public API to answer user questions with current, retrieved information.

#### Module 4: Designing Reliable Agent Workflows
- **Description:** Learners will design multi-step workflows that combine planning, tool use, memory, and human approval. The module emphasizes reliability, observability, safety, and responsible handling of agent actions.
- **Key Learning Outcomes:**
  1. Design multi-step agent workflows with clear state transitions.
  2. Add safeguards such as approval steps, permissions, and fallback behavior.
  3. Evaluate agent performance using logs, test cases, and quality metrics.
- **Interactive Assignment:** Design and test a workflow for an operations assistant that gathers information, proposes an action, and requests approval before execution.

#### Module 5: Final Project: Build and Evaluate an AI Agent
- **Description:** In the final module, learners will build a complete AI agent that solves a practical problem using multiple tools. They will document the design, demonstrate the workflow, and evaluate the agent for accuracy, reliability, and usability.
- **Key Learning Outcomes:**
  1. Develop an end-to-end AI agent with multiple tool integrations.
  2. Apply testing, error handling, safety controls, and evaluation methods.
  3. Present the agent’s architecture, capabilities, limitations, and results.
- **Interactive Assignment:** Build and present a complete tool-calling AI agent, such as a research assistant, scheduling assistant, support agent, or business operations helper.

---

### Assessment Plan

| Assessment Method | Description | Weightage (%) |
|---|---|---:|
| Module Quizzes | Short quizzes that assess concepts, terminology, and best practices from each module. | 15 |
| Practical Assignments | Hands-on exercises involving agent design, tool calling, API integration, and workflow testing. | 35 |
| Participation and Peer Review | Contributions to discussions, demonstrations, code reviews, and peer feedback activities. | 10 |
| Final Project | An end-to-end AI agent project, including implementation, documentation, testing, and presentation. | 40 |

---

### Certification Criteria
1. Achieve an overall course score of at least 70%.
2. Complete all module assignments and submit the final project.
3. Demonstrate a functional AI agent that uses at least two tools and includes basic error handling and safety controls.

---

### Marketing Strategy

#### Step 1: Define the Core Value Proposition
- Position the course as a practical introduction to building tool-enabled AI agents.
- Emphasize beginner-friendly instruction, hands-on projects, and real-world use cases.

#### Step 2: Identify and Reach Target Learners
- Promote the course to aspiring developers, automation professionals, analysts, and technical teams.
- Use communities focused on Python, APIs, generative AI, workflow automation, and developer tools.

#### Step 3: Create Educational Promotional Content
- Publish short tutorials demonstrating tool calling, API integration, and agent workflows.
- Share project previews, code snippets, learner outcomes, and common beginner challenges.

#### Step 4: Use Partnerships and Live Events
- Collaborate with developer communities, bootcamps, universities, and professional training organizations.
- Host free webinars or live demonstrations showing how to build a simple AI agent.

#### Step 5: Encourage Enrollment and Referrals
- Offer an introductory discount, early-bird pricing, or a free sample lesson.
- Provide referral incentives and showcase completed learner projects as social proof.

**Observation**

The model gave a full outline for "AI Agents and Tool Calling in Practice" in the same structure and formatting as the example:an overview with objectives and target audience, 5 modules, an assessment table, certification criteria and a 5-step marketing strategy. Every module has a description, exactly 3 learning outcomes and one interactive assignment, and the last module is a final project. The modules go in a logical order, from agent basics to tool calling, APIs and data, reliable workflows, and a final project. The assessment weights add up to 100% (15 + 35 + 10 + 40), with the final project the highest. The finish reason was "stop", so the outline was not cut off. Spelling out the exact structure, the number of modules, outcomes and steps, and the horizontal lines in the prompt gave a complete outline that follows the required format.

####